# 04 — Few-shot personalization

Spec section J: *"a lightweight few-shot personalization layer — e.g. a
small per-subject fine-tuning of the final regression layers (or a
feature-space affine recalibration) using only a handful of labeled
calibration segments per new subject"*.

The question it answers is a practical one: **how many cuff readings
does a new user have to provide before the model works for them?** The
answer is a curve over k = 0, 1, 3, 5, 10, 25, where k = 0 is the
uncalibrated number, so any drop is readable straight off it.

Spec section L: *"for a held-out subject, use only the first small % of
that subject's segments as calibration data and evaluate on the
remainder"*. That is `source='calfree'` — 144 unseen patients, 400 clips
each, the first k used to calibrate and the remaining ~395 scored.
Calibration clips are dropped from the metric.

## Setup

Run these cells in order before anything else. They work both on Colab
and on a laptop — on Colab the data is copied off Drive onto the local
disk first, because Drive is a network mount and the sampler reads clips
in random order, which would leave the GPU waiting on I/O. The copy also
stops immediately if the upload turned out to be incomplete.

In [ ]:
# Needs a GPU. Runtime > Change runtime type > T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU'

In [ ]:
import os, sys, glob, shutil, time

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content/drive')

if not IN_COLAB:
    # running from notebooks/ on a normal machine
    sys.path.insert(0, '../src')
    sys.path.insert(0, '../scripts')
else:
    from google.colab import drive
    drive.mount('/content/drive')

    # Find the bp folder wherever it was dropped in Drive.
    hits = glob.glob('/content/drive/MyDrive/**/data/colab_manifest.json',
                     recursive=True)
    if len(hits) != 1:
        raise SystemExit(f'expected exactly one bp folder, found {len(hits)}:'
                         + ''.join('\n  ' + h for h in hits))
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
    LOCAL_ROOT = '/content/bp'
    print('drive:', DRIVE_ROOT)

    # Copy the data to local disk first. Drive is a network mount and the
    # sampler reads clips in random order, so training straight off Drive
    # leaves the GPU idle waiting on I/O. A few minutes here saves hours.
    t0 = time.time()
    for rel in ['src', 'scripts', 'data/colab_manifest.json', 'data/features',
                'data/processed/pulsedb', 'data/processed/ppg_dalia/windows']:
        src, dst = f'{DRIVE_ROOT}/{rel}', f'{LOCAL_ROOT}/{rel}'
        if os.path.exists(dst):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dst)
        print(f'  copied {rel:40s} {time.time() - t0:5.0f}s')

    # results/ points back at Drive — metrics, predictions and checkpoints
    # all land under results/models/<name>/, so a dropped session does not
    # take the run with it.
    os.makedirs(f'{DRIVE_ROOT}/results/models', exist_ok=True)
    if not os.path.islink(f'{LOCAL_ROOT}/results'):
        os.symlink(f'{DRIVE_ROOT}/results', f'{LOCAL_ROOT}/results')

    os.environ['BP_ROOT'] = LOCAL_ROOT     # every path follows this
    sys.path.insert(0, LOCAL_ROOT + '/src')
    sys.path.insert(0, LOCAL_ROOT + '/scripts')

    # Fail now if the upload was incomplete, not twenty minutes into a run.
    from prepare_colab_upload import verify
    if verify(LOCAL_ROOT):
        raise SystemExit('copy is incomplete — see the list above')
    print(f'\nBP_ROOT = {LOCAL_ROOT}   (ready in {time.time() - t0:.0f}s)')

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set the runtime type and restart')

## First: is there anything for calibration to fix?

If the error were random noise, no amount of calibration could remove
it. If it is a per-subject offset, one reading is enough in principle.

`offset_headroom` measures which it is. It hands every subject its own
best possible constant offset — fitted on that subject's own test clips,
so it is a ceiling nobody can reach — and reports how far the error
falls. That gives the scale to read the real gain against.

In [ ]:
from train import personalize

MODEL = 'gb'   # 'gb' | 'rf' | 'cnn' | 'resnet' | 'transformer'

# A deep model is trained from scratch here (hours), because the
# calibration clips and the test clips have to be scored by the *same*
# model — calibrating one model and testing another measures nothing.
cal = personalize(model=MODEL, source='calfree')

## The curve

Three ways of choosing which clips a subject calibrates on, reported side
by side rather than one being asserted: `first` (the spec's literal
reading, and what a device could actually collect), `random`, and
`spread` (readings covering that person's own BP range).

In [ ]:
import pandas as pd

rows = []
for pick, curve in cal['curves'].items():
    for row in curve:
        for target in ('sbp', 'dbp'):
            m = row['results'][target]
            rows.append(dict(pick=pick, shots=row['shots'], target=target,
                             n=row['n_scored'], mae=m['mae'], sde=m['sde'],
                             aami='PASS' if m['aami_pass'] else 'FAIL'))
df = pd.DataFrame(rows)
display(df[df.target == 'sbp'].pivot(index='shots', columns='pick',
                                     values='mae').round(2))
df.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, target in enumerate(('sbp', 'dbp')):
    d = df[df.target == target]
    for pick in d['pick'].unique():
        s = d[d['pick'] == pick]
        ax[i].plot(s.shots, s.mae, 'o-', label=pick)
    ax[i].axhline(5, ls='--', c='#c0392b', label='clinical standard')
    ax[i].set_title(target.upper())
    ax[i].set_xlabel('calibration readings per subject (k)')
    ax[i].set_ylabel('MAE (mmHg)')
    ax[i].grid(alpha=.3)
    ax[i].legend()
plt.tight_layout()

## Secondary: PulseDB's own AAMI subsets

This gives a different answer, and the reason is in the data rather than
in the method. `aami_test` is built to span the full BP range on purpose,
so its clips average 134.9 mmHg SBP while the first five `aami_cal` clips
of the same patients average 119.7. A first-k calibration therefore bakes
that -15 mmHg gap into every prediction and makes the error worse.

Worth reporting: where the calibration readings come from is itself a
design decision, not a detail.

In [ ]:
aami = personalize(model=MODEL, source='aami')

In [ ]:
# Everything this session produced, on Drive
from paths import MODEL_RESULTS
for p in sorted(MODEL_RESULTS.glob('*/*.json')):
    print(f'{p.stat().st_size / 1e3:8.1f} kB  {p.name}')